%md
# Notebook 06 – Model Inference

## Why Model Inference?

After a machine learning model has been trained and registered, hospitals need to score new patient records without retraining the model.

The Model Inference layer loads the latest production model from the Unity Catalog Model Registry, predicts readmission risk for new patients, and stores the prediction results for dashboards and clinical decision support.

### Objective

The objective of this notebook is to perform batch inference using the registered Champion model and generate readmission risk predictions for newly ingested patient data.

### Input Tables

This notebook reads the following Gold tables:

- gold.patient_cohorts
- gold.encounter_summaries
- gold.readmission_within_30days

It also loads the registered Champion model from the Unity Catalog Model Registry.

### Output

This notebook produces:

- gold.readmission_predictions
- Predicted readmission probability
- Predicted risk category (Low / Medium / High)
- Batch inference results for reporting and dashboards

In [0]:
%run ./00_UnityCatalog_Bootstrap_&_Audit_log

In [0]:
from datetime import datetime

start_time = datetime.now()

In [0]:
import mlflow
import mlflow.pyfunc
import pandas as pd
import numpy as np

from pyspark.sql.functions import max

from pyspark.ml.feature import (
    StringIndexerModel,
    OneHotEncoderModel,
    VectorAssembler
)

%md
### 1. Load Registered Champion Model

The latest production-ready model is loaded from the Unity Catalog Model Registry using the **Champion** alias.

This ensures that inference always uses the currently deployed model without requiring any code changes.

In [0]:
model_uri = "models:/Readmission_XGBoost_Model@Champion"

model = mlflow.pyfunc.load_model(model_uri)

In [0]:
gold_df = spark.table(
    "db_healthcare_kl.gold.ml_readmission_features"
)

### 2. Load New Patient Records

The notebook loads new patient records from the Gold layer that have not yet been scored.

Only unseen patients are selected for inference to avoid generating duplicate predictions.

In [0]:
latest_time = (
    gold_df
    .select(max("processed_timestamp"))
    .collect()[0][0]
)

new_patients = gold_df.filter(
    gold_df.processed_timestamp == latest_time
)

In [0]:
feature_df = new_patients.drop(
    "person_id",
    "processed_timestamp",
    "readmission_30day_flag"
)

In [0]:
visit_indexer = StringIndexerModel.load(
    "/Volumes/db_healthcare_kl/default/ml_models/visit_indexer"
)

diagnosis_indexer = StringIndexerModel.load(
    "/Volumes/db_healthcare_kl/default/ml_models/diagnosis_indexer"
)

cohort_indexer = StringIndexerModel.load(
    "/Volumes/db_healthcare_kl/default/ml_models/cohort_indexer"
)

In [0]:
feature_df = visit_indexer.transform(feature_df)
feature_df = diagnosis_indexer.transform(feature_df)
feature_df = cohort_indexer.transform(feature_df)

In [0]:
visit_encoder = OneHotEncoderModel.load(
    "/Volumes/db_healthcare_kl/default/ml_models/visit_encoder"
)

diagnosis_encoder = OneHotEncoderModel.load(
    "/Volumes/db_healthcare_kl/default/ml_models/diagnosis_encoder"
)

cohort_encoder = OneHotEncoderModel.load(
    "/Volumes/db_healthcare_kl/default/ml_models/cohort_encoder"
)

%md
### 3. Prepare Model Features

The required machine learning features are extracted and transformed into the same format used during model training.

This guarantees consistency between training and inference.

In [0]:
feature_df = visit_encoder.transform(feature_df)
feature_df = diagnosis_encoder.transform(feature_df)
feature_df = cohort_encoder.transform(feature_df)

In [0]:
assembler = VectorAssembler.load(
    "/Volumes/db_healthcare_kl/default/ml_models/assembler"
)

In [0]:
feature_df = assembler.transform(feature_df)

In [0]:
pdf = feature_df.select("features").toPandas()

In [0]:
feature_names = [f"feature_{i}" for i in range(46)]

X = pd.DataFrame(
    np.vstack(
        pdf["features"].apply(lambda x: x.toArray())
    ),
    columns=feature_names
)

%md
### 4. Generate Readmission Predictions

The registered Champion model predicts:

- Readmission Probability
- Predicted Readmission Class

for every new patient record.

In [0]:
predictions = model.predict(X)

In [0]:
result = new_patients.toPandas()

result["prediction"] = predictions

In [0]:
prediction_df = spark.createDataFrame(result)

In [0]:
prediction_df.write \
    .mode("overwrite") \
    .saveAsTable(
        "db_healthcare_kl.gold.readmission_predictions"
    )

%md
### 5. Assign Risk Categories

Prediction probabilities are converted into clinical risk categories.

Example:

- Low Risk
- Medium Risk
- High Risk

These categories simplify interpretation for clinicians and hospital administrators.

In [0]:
from pyspark.sql.functions import when

prediction_df = prediction_df.withColumn(
    "risk",
    when(prediction_df.prediction == 1, "High Risk")
    .otherwise("Low Risk")
)

display(prediction_df)

In [0]:
end_time = datetime.now()

record_count = prediction_df.count()

log_pipeline_run(
    pipeline_name="Healthcare Lakehouse",
    layer="Bronze",
    status="SUCCESS",
    records_processed=record_count,
    start_time=start_time,
    end_time=end_time
)